# 03. 시설물 파손 분류 모델 학습 (Colab GPU) — 계획 2.7

가드레일·표지판 크롭을 **normal / damaged** 로 나누는 YOLO11n-cls 를 학습하고, **ONNX + 모델 카드
(`damage_cls_<버전>.onnx` + `.json`)** 를 Drive `models/` 에 저장한다. 블랙박스에서는
`OnnxDetector(..., damage_model="ai/models/damage_cls_v1.onnx")` 가 guardrail·sign 탐지마다 `damage`
(`"suspected"`/`"normal"`)와 `damage_conf`(= P(damaged))를 채운다. 결과는 **'파손 의심'일 뿐 확정이 아니다**
(PRD FR-1.2, 신고 제안은 여러 차량이 같은 판단일 때만 FR-7.2).

- 데이터: Drive `parts/krid.zip` 의 `cls/` (Mac 에서 `prepare_local` 로 만든 KRID 크롭). damaged 가 매우 적어서
  (train 107 / val 15 / test 17) normal 을 해시로 골라 **train 5배, val·test 10배**만 쓴다.
- 크롭은 추론과 똑같이 만든다: `facility_crop`(CropWriter 와 같은 함수) → `square_input`(비율 유지 정사각형 224).
- 임계값은 **val 에서 F1 최대**인 P(damaged) 로 정하고, test 에서 precision·recall·F1·normal 오탐률을
  시설물 종류별로 낸다.

**미리 할 것**: **런타임 → 런타임 유형 변경 → T4 GPU**. 세션이 끊겨도 Drive 의 `last.pt` 에서 이어진다
(위에서부터 다시 실행).

In [ ]:
# ===== 설정 =====
DRIVE_ROOT = "/content/drive/MyDrive/daboyeo_data"
REPO_URL = "https://github.com/ParkGyeom/daboyeo.git"

VERSION = "v1"
BASE_MODEL = "yolo11n-cls.pt"   # 블랙박스 CPU 에서 탐지기 뒤에 붙으므로 n급
EPOCHS = 60
PATIENCE = 15                   # val 정확도가 이만큼 안 오르면 조기 종료
IMGSZ = 224                     # 크롭 긴 변 224 (CropWriter) 와 같게
BATCH = 64
NORMAL_RATIO = {"train": 5, "val": 10, "test": 10}   # normal 을 damaged 의 몇 배까지 쓸지
AUTO_AUGMENT = "randaugment"    # 회전(±약 9°)·밝기·대비·색 등. None 이면 아래 HSV 만 (회전 없음)
SCALE = 0.3                     # 무작위 확대 자르기: 원본 면적의 70~100%
HSV = dict(hsv_h=0.015, hsv_s=0.4, hsv_v=0.4)        # AUTO_AUGMENT=None 일 때만 쓰인다 (Ultralytics 규칙)
FULL_NORMAL_FPR = True          # test 의 normal 크롭 전부(약 4,800장)로 오탐률도 따로 잰다 (CPU 몇 분)

RUNS = f"{DRIVE_ROOT}/runs"
MODELS = f"{DRIVE_ROOT}/models"
RUN_NAME = f"damage_cls_{VERSION}"
DATASET = "/content/dataset"    # krid.zip 의 cls/ 를 풀 곳 (Colab 로컬 디스크)
CLS_ROOT = f"{DATASET}/dmg_{VERSION}"   # normal 을 줄인 학습용 폴더 (Ultralytics classify 구조)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, pathlib, json, shutil, subprocess, sys, time, zipfile
for d in (RUNS, MODELS):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU 없음 — 런타임 유형을 T4 GPU 로 바꿔야 한다"

### 코드 설치 (`ultralytics` + 레포)

In [ ]:
REPO = "/content/daboyeo"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)   # 항상 최신 코드
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
!pip install -q ultralytics onnx onnxslim onnxruntime -e {REPO}/contracts -e {REPO}/ai
sys.path[:0] = [f"{REPO}/contracts", f"{REPO}/ai"]
from daboyeo_ai.data.damage_cls import crop_type, read_type_table, subsample_cls
from daboyeo_ai.eval.damage_metrics import compare_probs, pick_threshold, report_by_group, score_folder
from daboyeo_ai.inference import DamageClassifier
types_table = read_type_table()   # 크롭 → guardrail/sign (Mac 에서 라벨로 만든 표, 레포에 있음)
print("type table images:", len(types_table))

## 1. 데이터 풀기 + normal 줄이기

`krid.zip` 에서 **`cls/` 만** 푼다 (탐지용 `det/` 는 02 노트북). 그다음 damaged 는 전부, normal 은
`NORMAL_RATIO` 배만 해시로 골라 정사각형 PNG 로 `CLS_ROOT` 에 쓴다 (다시 실행해도 같은 파일).

In [ ]:
from collections import Counter
import pandas as pd

krid_zip = pathlib.Path(DRIVE_ROOT) / "parts" / "krid.zip"
assert krid_zip.exists(), f"{krid_zip} 없음 — Mac 에서 만든 krid.zip 을 Drive parts/ 에 올려야 한다"
src = pathlib.Path(DATASET) / "cls"
marker = pathlib.Path(DATASET) / ".done_krid_cls"
if not marker.exists():
    with zipfile.ZipFile(krid_zip) as zf:
        zf.extractall(DATASET, [m for m in zf.namelist() if m.startswith("cls/")])
    marker.touch()

sub = subsample_cls(src, pathlib.Path(CLS_ROOT), NORMAL_RATIO)
display(pd.DataFrame(sub).T)

def type_counts(root, ext):
    c = Counter()
    for f in pathlib.Path(root).glob(f"*/*/*.{ext}"):
        c[(f.parent.parent.name, f.parent.name, crop_type(f.stem, types_table) or "unknown")] += 1
    return pd.Series(c).unstack([1, 2]).fillna(0).astype(int)

print("전체 크롭 (시설물 종류별)")
display(type_counts(src, "jpg"))
print("학습·평가에 쓰는 크롭")
counts = type_counts(CLS_ROOT, "png")
display(counts)

## 2. 학습

- 처음: `BASE_MODEL`(ImageNet 사전학습)에서 시작. 다시 실행: Drive `runs/damage_cls_<버전>/weights/last.pt` 에서 **이어서**
- 증강: 좌우 반전 O, 상하 반전 X, `SCALE`(살짝 확대), `AUTO_AUGMENT`(작은 회전·밝기·색)
- Ultralytics 는 val **정확도**(top-1)로 best.pt 를 고른다. 파손 판정 기준(임계값)은 다음 단계에서 따로 정한다.

In [ ]:
from ultralytics import YOLO
import torch

run_dir = pathlib.Path(RUNS) / RUN_NAME
last = run_dir / "weights" / "last.pt"

def is_finished(ckpt_path):
    # 끝난 학습의 체크포인트는 epoch=-1, optimizer=None 으로 정리된다.
    # 이걸 resume 하면 Ultralytics 가 경고만 하고 새 학습을 시작하므로 미리 막는다 (02 와 같음).
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    return ck.get("epoch", -1) == -1 or ck.get("optimizer") is None

t = time.time()
if last.exists() and is_finished(last):
    print(f"{RUN_NAME} 학습은 이미 끝났다 — 건너뜀 (다시 학습하려면 VERSION 을 바꾸거나 {run_dir} 를 지운다)")
elif last.exists():
    print("중단된 학습을 이어서:", last)
    YOLO(str(last)).train(resume=True)
else:
    YOLO(BASE_MODEL).train(
        data=CLS_ROOT, epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
        project=RUNS, name=RUN_NAME, exist_ok=True, seed=0, workers=2,
        fliplr=0.5, flipud=0.0, scale=SCALE, auto_augment=AUTO_AUGMENT, erasing=0.0, **HSV, plots=True,
    )
print(f"{(time.time() - t) / 60:.1f} 분")

## 3. ONNX 변환 → val 임계값 → test 평가

평가는 **블랙박스와 같은 경로**(ONNX + 우리 `DamageClassifier` 전처리)로 한다. 클래스 순서는 학습 폴더 이름
순서(`damaged`, `normal`)를 모델에서 읽어 카드에 적는다 — 코드에서 0번이 damaged 라고 가정하지 않는다.

In [ ]:
best_pt = run_dir / "weights" / "best.pt"
best = YOLO(str(best_pt))
classes = [best.names[i] for i in range(len(best.names))]
print("classes (출력 순서):", classes)
onnx_path = pathlib.Path(best.export(format="onnx", imgsz=IMGSZ, opset=17, simplify=True, dynamic=False))

CROP = {"pad": 0.1, "resize_max": 224, "min_side_px": 24}   # CropWriter 기본값과 같아야 한다
card_path = onnx_path.with_suffix(".json")
card_path.write_text(json.dumps({"classes": classes, "threshold": 0.5, "crop": CROP}))
clf = DamageClassifier(onnx_path, threads=2)

scores = {s: score_folder(CLS_ROOT, s, clf.prob_damaged_crop) for s in ("val", "test")}
names_v, y_v, p_v = scores["val"]
THRESHOLD, val_f1 = pick_threshold(y_v, p_v)
print(f"val 최대 F1 임계값 = {THRESHOLD:.4f} (F1 {val_f1:.3f})")

def by_type(split):
    names, y, p = scores[split]
    return report_by_group(y, p, THRESHOLD, [crop_type(n, types_table) for n in names])

metrics = {"val": by_type("val"), "test": by_type("test")}
for split in ("val", "test"):
    print(f"\n[{split}]  threshold={THRESHOLD:.4f}")
    display(pd.DataFrame(metrics[split]).T)

t_all = metrics["test"]["all"]
print("\ntest 혼동 행렬 (행=정답, 열=예측)")
display(pd.DataFrame([[t_all["tp"], t_all["fn"]], [t_all["fp"], t_all["tn"]]],
                     index=["정답 damaged", "정답 normal"], columns=["예측 suspected", "예측 normal"]))

In [ ]:
# test normal 크롭 전부로 오탐률 (실제 도로는 거의 다 정상이라 이 값이 '의심' 남발 여부를 보여준다)
import numpy as np
full_normal_fpr = None
if FULL_NORMAL_FPR:
    t = time.time()
    names_all, y_all, p_all = score_folder(src, "test", clf.prob_damaged_crop, exts=(".jpg",))
    is_normal = y_all == 0
    p_n = p_all[is_normal]
    types_n = np.array([crop_type(n, types_table) or "unknown" for n, ok in zip(names_all, is_normal) if ok])
    full_normal_fpr = {}
    for k in ("all", "guardrail", "sign"):
        m = np.ones(len(p_n), bool) if k == "all" else types_n == k
        full_normal_fpr[k] = {"n": int(m.sum()), "fpr": round(float((p_n[m] >= THRESHOLD).mean()), 4) if m.any() else None}
    print(full_normal_fpr, f"({time.time() - t:.0f} s)")

## 4. 일치 확인 + CPU 속도

같은 ONNX 를 **Ultralytics 로 돌린 P(damaged)** 와 우리 `DamageClassifier` 결과를 test 크롭 전부에서 비교한다.
`max_abs_diff` 가 0.01 미만, `label_agree` 가 1.0 이면 통과.

In [ ]:
import cv2
ref_model = YOLO(str(onnx_path), task="classify")
d_idx = classes.index("damaged")
names_t, y_t, p_t = scores["test"]
files_t = [pathlib.Path(CLS_ROOT) / "test" / ("damaged" if yy else "normal") / f"{n}.png" for n, yy in zip(names_t, y_t)]
p_ref = [float(r.probs.data[d_idx]) for f in files_t
         for r in ref_model.predict(str(f), imgsz=IMGSZ, verbose=False)]
parity = compare_probs(p_t, p_ref, THRESHOLD)
print(parity)
if parity["max_abs_diff"] is None or parity["max_abs_diff"] >= 0.01 or parity["label_agree"] < 1.0:
    print("⚠️ DamageClassifier 결과가 Ultralytics 와 다르다 — 이 출력을 공유할 것")

In [ ]:
# CPU 속도 (크롭 1장당, 크롭 자르기 + 분류). Colab CPU 기준 참고값
frame = cv2.imread(str(files_t[0]))
speed = {}
for threads in (1, 2):
    c = DamageClassifier(onnx_path, threads=threads)
    c.prob_damaged_crop(frame)  # 워밍업
    t = time.perf_counter()
    for _ in range(50):
        c.prob_damaged(frame, (10, 10, 200, 120))
    speed[f"{threads}thread_ms_per_crop"] = round((time.perf_counter() - t) / 50 * 1000, 2)
print(speed)

## 5. Drive `models/` 에 저장

In [ ]:
import datetime
out_onnx = pathlib.Path(MODELS) / f"damage_cls_{VERSION}.onnx"
shutil.copy(onnx_path, out_onnx)
shutil.copy(best_pt, pathlib.Path(MODELS) / f"damage_cls_{VERSION}.pt")
results_csv = run_dir / "results.csv"
card = {
    "version": VERSION,
    "file": out_onnx.name,
    "task": "guardrail/sign crop damage classification (normal vs damaged)",
    "classes": classes,                     # 출력 순서. DamageClassifier 는 이걸 읽는다
    "threshold": THRESHOLD,                 # P(damaged) >= threshold → damage="suspected"
    "threshold_rule": "max F1 on val",
    "input_size": IMGSZ,
    "input": "1x3xHxW float32 RGB 0~1 — facility_crop(pad, resize_max) → square_input(letterbox 114)",
    "crop": CROP,
    "applies_to": ["guardrail", "sign"],
    "base_model": BASE_MODEL,
    "normal_ratio": NORMAL_RATIO,
    "augment": {"fliplr": 0.5, "flipud": 0.0, "scale": SCALE, "auto_augment": AUTO_AUGMENT, **HSV},
    "data_counts": {f"{s}/{l}/{t}": int(v) for (l, t), col in counts.items() for s, v in col.items()},
    "subsample": sub,
    "epochs_trained": len(pd.read_csv(results_csv)) if results_csv.exists() else None,
    "metrics": metrics,                     # val / test, all + guardrail + sign
    "test_full_normal_fpr": full_normal_fpr,
    "onnx_parity": parity,
    "colab_cpu_speed": speed,
    "meaning": "suspected = 파손 의심 (확정 아님). 신고 제안은 여러 차량 합의 시만 (PRD FR-7.2)",
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
}
(pathlib.Path(MODELS) / f"damage_cls_{VERSION}.json").write_text(json.dumps(card, ensure_ascii=False, indent=1))
print(json.dumps({k: card[k] for k in ("file", "classes", "threshold", "epochs_trained", "onnx_parity", "colab_cpu_speed")},
                 ensure_ascii=False, indent=1))
print(json.dumps(metrics["test"]["all"], indent=1))
!ls -la {MODELS}

## 다음

1. Drive `daboyeo_data/models/damage_cls_<버전>.onnx` 와 `.json` 을 노트북 PC 의 `ai/models/` 에 **둘 다** 내려받는다
   (`.json` 이 없으면 클래스 순서·임계값을 몰라서 열리지 않는다. `*.onnx` 는 git 에 올리지 않는다).
2. 위 test 표(시설물별)와 `damage_cls_<버전>.json` 을 공유한다 → Go/No-go (PRD 11/20).
3. 정현우: `OnnxDetector(..., damage_model="ai/models/damage_cls_v1.onnx")` (docs/ai-handoff.md).